# 🎵 PHÂN TÍCH CHUYÊN SÂU DATASET EVENTSIM
Sau khi dữ liệu Spotify từ Kafka đã được lưu vào HDFS, ta sẽ thực hiện 2 bài toán: 
1. **WordCount** trên toàn bộ mớ dữ liệu thô (JSON).
2. Tìm **Top 5 bài hát** được nghe nhiều nhất.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, lower, regexp_replace, count, length
from pyspark.sql.types import StructType, StringType, DoubleType, LongType
import os
import json
import pandas as pd

In [2]:

os.environ["HADOOP_USER_NAME"] = "hadoop"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc, explode, split

spark = SparkSession.builder.appName("EventSim-Analysis").getOrCreate()
print("Spark đã sẵn sàng để bóc tách dữ liệu!")

Spark đã sẵn sàng để bóc tách dữ liệu!


## 1. Bài toán 1: Đọc dữ liệu từ HDFS và Linecount toàn bộ dataset
Dữ liệu được lấy từ thư mục lưu trữ Data Lake.

In [37]:
silver_df = spark.read.parquet("hdfs://master:9000/datalake/silver/eventsim")
# --- KIỂM TRA TỔNG SỐ DÒNG (LINE COUNT) ---
total_lines = silver_df.count()
# --- TẠO BẢNG TỔNG KẾT ---
# Mình sẽ gom các chỉ số vào một bảng để bạn chụp ảnh 1 lần
summary_data = {
    "Chỉ số giám sát": ["Tổng số sự kiện (Line Count)", "Số người dùng độc bản", "Số lỗi phát hiện"],
    "Kết quả": [f"{total_lines:,} dòng", 
                 f"{silver_df.select('userId').distinct().count():,} người", 
                 f"{silver_df.filter('status != 200').count():,}"
    ]
}
# Hiển thị bảng tóm tắt
print("✅ BÁO CÁO GIÁM SÁT HỆ THỐNG")
summary_table = pd.DataFrame(summary_data)
display(summary_table)
# --- HIỂN THỊ CHI TIẾT THEO HÀNH ĐỘNG ---
print("\n✅ CHI TIẾT LƯU LƯỢNG THEO TRANG (PAGE ANALYSIS)")
page_stats = silver_df.groupBy("page").count().orderBy("count", ascending=False).toPandas()
display(page_stats)

✅ BÁO CÁO GIÁM SÁT HỆ THỐNG


,Chỉ số giám sát,Kết quả
0,Tổng số sự kiện (Line Count),"297,865 dòng"
1,Số người dùng độc bản,946 người
2,Số lỗi phát hiện,"8,130"



✅ CHI TIẾT LƯU LƯỢNG THEO TRANG (PAGE ANALYSIS)


,page,count
0,NextSong,248895
1,Home,32465
2,Logout,3545
3,Login,3535
4,Help,2132
5,Downgrade,1925
6,Settings,1850
7,About,1318
8,Upgrade,1145
9,Save Settings,390


In [33]:
first_row = silver_df.limit(1).toJSON().first()
print(json.dumps(json.loads(first_row), indent=4))


{
    "ts": 1774415505113,
    "userId": "10",
    "sessionId": 2149,
    "auth": "Logged In",
    "method": "PUT",
    "status": 200,
    "level": "free",
    "itemInSession": 0,
    "location": "Laurel, MS",
    "userAgent": "\"Mozilla/5.0 (iPhone; CPU iPhone OS 7_1_2 like Mac OS X) AppleWebKit/537.51.2 (KHTML, like Gecko) Version/7.0 Mobile/11D257 Safari/9537.53\"",
    "lastName": "Berg",
    "firstName": "Jonathan",
    "registration": 1774273697113,
    "gender": "M",
    "artist": "Sam Cooke",
    "song": "Ain't Misbehavin",
    "ingestion_time": "2026-04-07T14:17:54.109Z",
    "event_time": "2026-03-25T05:11:45.000Z",
    "registration_time": "2026-03-23T13:48:17.000Z",
    "city": "Laurel",
    "state": "MS",
    "browser": "Safari",
    "os": "iPhone",
    "page": "NextSong"
}


## 2. Bài toán 1: WordCount trên toàn bộ Dataset
Coi toàn bộ các bản ghi JSON như một văn bản và đếm các từ khóa xuất hiện.

In [39]:
gold_path = "hdfs://master:9000/datalake/gold/song_rankings"
print("🚀 Đang tải dữ liệu từ Gold Layer...")
try:
    # 3. Đọc dữ liệu Parquet
    gold_df = spark.read.parquet(gold_path)
    # 4. Lấy Top 10 bài hát hot nhất
    # Sắp xếp theo lượt nghe giảm dần
    songs = gold_df.orderBy("play_count", ascending=False)
    
    # Chuyển sang Pandas để Jupyter hiển thị bảng HTML đẹp mắt
    report_table = songs.toPandas()
    # 5. Hiển thị kết quả
    print("\n✅ Tổng số bài hát được user nghe (xếp từ nghe nhiều nhất đến thấp nhất)")
    display(report_table)
except Exception as e:
    print(f"❌ Lỗi: {e}")
    print("\nGợi ý: Hãy đảm bảo bạn đã chạy Medallion DAG trong Airflow thành công ít nhất 1 lần nhé!")

🚀 Đang tải dữ liệu từ Gold Layer...

✅ Tổng số bài hát được user nghe (xếp từ nghe nhiều nhất đến thấp nhất)


,artist,song,play_count
0,Dwight Yoakam,You're The One,1410
1,BjÃÂ¶rk,Undo,1010
2,Kings Of Leon,Revelry,915
3,Harmonia,Sehr kosmisch,735
4,Barry Tuckwell/Academy of St Martin-in-the-Fie...,Horn Concerto No. 4 in E flat K495: II. Romanc...,690
...,...,...,...
22763,Los Traileros Del Norte,Supuestamente Gane,5
22764,Flight Of The Conchords,Fashion Is Danger,5
22765,Everclear,Science Fiction,5
22766,Jason Forrest,Why I Love ELO,5


## 3. Bài toán 2: Truy vấn TOP 10 BÀI HÁT
Tìm bảng xếp hạng 5 bài hát hot nhất dựa trên cột `song`.

In [41]:
gold_path = "hdfs://master:9000/datalake/gold/song_rankings"
print("🚀 Đang tải dữ liệu từ Gold Layer...")
try:
    # 3. Đọc dữ liệu Parquet
    gold_df = spark.read.parquet(gold_path)
    # 4. Lấy Top 10 bài hát hot nhất
    # Sắp xếp theo lượt nghe giảm dần
    songs = gold_df.orderBy("play_count", ascending=False).limit(10)
    
    # Chuyển sang Pandas để Jupyter hiển thị bảng HTML đẹp mắt
    report_table = songs.toPandas()
    # 5. Hiển thị kết quả
    print("\n✅ BẢNG XẾP HẠNG TOP 10 BÀI HÁT (STREAMLIFY REPORT)")
    display(report_table)
except Exception as e:
    print(f"❌ Lỗi: {e}")
    print("\nGợi ý: Hãy đảm bảo bạn đã chạy Medallion DAG trong Airflow thành công ít nhất 1 lần nhé!")

🚀 Đang tải dữ liệu từ Gold Layer...

✅ BẢNG XẾP HẠNG TOP 10 BÀI HÁT (STREAMLIFY REPORT)


,artist,song,play_count
0,Dwight Yoakam,You're The One,1410
1,BjÃÂ¶rk,Undo,1010
2,Kings Of Leon,Revelry,915
3,Harmonia,Sehr kosmisch,735
4,Barry Tuckwell/Academy of St Martin-in-the-Fie...,Horn Concerto No. 4 in E flat K495: II. Romanc...,690
5,Florence + The Machine,Dog Days Are Over (Radio Edit),670
6,OneRepublic,Secrets,535
7,Five Iron Frenzy,Canada,505
8,Kings Of Leon,Use Somebody,495
9,Tub Ring,Invalid,475


In [42]:
# df_final = df_final.filter(col("song").isNotNull())
# song_counts = df_final.groupBy("song", "artist") \
#     .agg(count("*").alias("play_count")) \
#     .orderBy(col("play_count").desc()) 
# print("Top 10 bài hát được nghe nhiều nhất:")
# song_counts.show(10, truncate=100)

## 4. Lưu kết quả Báo cáo (Tầng Gold)
Lưu lại TOP 5 để phục vụ Dashboard hoặc thuyết trình.

In [20]:
output_path = "hdfs://master:9000/datalake/output/eventsim/top_5_songs/"
song_counts.write.mode("overwrite").parquet(output_path)
print(f"Đã lưu kết quả vào: {output_path}")
print("Soi thử báo cáo từ HDFS:")
spark.read.parquet(output_path).show()

Đã lưu kết quả vào: hdfs://master:9000/datalake/output/eventsim/top_5_songs/
Soi thử báo cáo từ HDFS:
+--------------------+--------------------+----------+
|                song|              artist|play_count|
+--------------------+--------------------+----------+
|      You're The One|       Dwight Yoakam|       282|
|                Undo|            BjÃÂ¶rk|       202|
|             Revelry|       Kings Of Leon|       183|
|       Sehr kosmisch|            Harmonia|       147|
|Horn Concerto No....|Barry Tuckwell/Ac...|       138|
|Dog Days Are Over...|Florence + The Ma...|       134|
|             Secrets|         OneRepublic|       107|
|              Canada|    Five Iron Frenzy|       101|
|        Use Somebody|       Kings Of Leon|        99|
|             Invalid|            Tub Ring|        95|
|Catch You Baby (S...|       Lonnie Gordon|        87|
|    Ain't Misbehavin|           Sam Cooke|        85|
|SinceritÃÂ© Et J...|     Alliance Ethnik|        78|
|    Bring Me To L